## Gold Cascade Modeling (Deliverable 1.3.3)

This notebook implements the full three-stage cascade anomaly detection pipeline. The objective is to determine whether a layered approach improves alert quality when compared to the single-model baseline.

**Purpose:**  
To operationalize the cascade architecture defined in Section C, generating alert outputs for each stage of the cascade: broad detection (Stage 1), refined detection (Stage 2), and rule/profile/historical confirmation (Stage 3).

**Stages Implemented:**

1. **Stage 1 — Broad Isolation Forest**  
   High-sensitivity anomaly screen using all 50 Gold numeric features.

2. **Stage 2 — Narrow Isolation Forest**  
   Secondary detector trained on the reduced feature subset identified during Silver EDA.

3. **Stage 3 — Rule / Profile / Historical Confirmation**  
   Final confirmation based on behavior profiles, persistence checks, drift features, and cross-sensor consistency.

**Key Goals:**

- Load the Gold preprocessed dataset and Gold feature artifacts.
- Train Stage 1 and Stage 2 Isolation Forest models.
- Apply Stage 3 rule/profile confirmation logic based on Silver EDA outputs.
- Generate and store alert outputs for all three cascade stages.
- Export all cascade artifacts for comparative evaluation.

**Relevance to Section C:**  
This notebook produces the layered alert outputs required for evaluating the cascade’s effect on false positives, noisy alerts, and anomaly sensitivity. These outputs are necessary for the statistical tests, alert-volume comparisons, and visual communication described in Section C.

## Cascade Modeling Setup and Imports

In this section I am loading the libraries and project utilities needed for the Gold cascade modeling stage.

The purpose here is to get the notebook ready before I start working with the Gold modeling inputs. That includes:
- standard Python libraries
- model and metrics utilities
- path and config loading
- logging
- truth-record helpers
- artifact saving utilities
- experiment tracking support

At this point I am not fitting the cascade yet. I am just preparing the notebook so the full multi-stage workflow can run in a structured and repeatable way.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

from pathlib import Path
import yaml

import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import (
    train_test_split, 
    KFold,
)

from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler, 
    OneHotEncoder, 
    RobustScaler,
)

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import (
    RandomForestClassifier, 
    IsolationForest,
)

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_recall_fscore_support, 
    roc_auc_score, 
    average_precision_score,
)

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import (
    load_data, 
    save_data, 
    save_json, 
    load_json,
)

from utils.eda_logging import profile_dataframe

from utils.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.wandb_utils import finalize_wandb_stage

from utils.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.core.cascade_row_tracking import (
    ensure_stable_row_id,
    build_stage_scoring_frame,
    score_isolation_forest_stage,
    merge_stage_results_back,
    finalize_stage_flag_columns,
    get_detected_rows_dataframe,
    get_stage_detected_rows_dataframe,
)

from utils.postgres_util import get_engine_from_env
from utils.layer_postgres_writer import (
    write_layer_dataframe, 
    prepare_layer_dataframe,
)


# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

## Cascade Modeling Setup and Imports

In this section I am loading the libraries and project utilities needed for the Gold cascade modeling stage.

The purpose here is to get the notebook ready before I start working with the Gold modeling inputs. That includes:
- standard Python libraries
- model and metrics utilities
- path and config loading
- logging
- truth-record helpers
- artifact saving utilities
- experiment tracking support

At this point I am not fitting the cascade yet. I am just preparing the notebook so the full multi-stage workflow can run in a structured and repeatable way.

In [ ]:
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_ROOT = paths.configs
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

CASCADE_VARIANT = "default"

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="gold_cascade",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

GOLD_CFG = CONFIG["gold_cascade"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)

TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE

# Stage details
STAGE = "gold"
LAYER_NAME = GOLD_CFG["layer_name"]
GOLD_VERSION = CONFIG["versions"]["gold"]
RECIPE_ID = GOLD_CFG["recipe_id"]
TRUTH_VERSION = CONFIG["versions"]["truth"]
PIPELINE_MODE = PIPELINE["execution_mode"]
RUN_MODE = CONFIG["runtime"]["mode"]

DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

GOLD_PROCESS_RUN_ID = make_process_run_id(GOLD_CFG["process_run_id_prefix"])



# Weights and Biases
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{GOLD_VERSION}"

# File names
GOLD_PREPROCESSED_FILE_NAME = FILENAMES["gold_preprocessed_file_name"]
GOLD_PREPROCESSED_SCALED_FILE_NAME = FILENAMES["gold_preprocessed_scaled_file_name"]

GOLD_FIT_FILE_NAME = FILENAMES["gold_fit_file_name"]
GOLD_TRAIN_FILE_NAME = FILENAMES["gold_train_file_name"]
GOLD_TEST_FILE_NAME = FILENAMES["gold_test_file_name"]

STAGE1_FEATURES_FILE_NAME = FILENAMES["stage1_features_file_name"]
STAGE2_FEATURES_FILE_NAME = FILENAMES["stage2_features_file_name"]
STAGE3_PRIMARY_FILE_NAME = FILENAMES["stage3_primary_file_name"]
STAGE3_SECONDARY_FILE_NAME = FILENAMES["stage3_secondary_file_name"]

STAGE1_MODEL_FILE_NAME = FILENAMES["cascade_defaults_stage1_model_file_name"]
STAGE2_MODEL_FILE_NAME = FILENAMES["cascade_defaults_stage2_model_file_name"]

CASCADE_THRESHOLDS_FILE_NAME = FILENAMES["cascade_defaults_thresholds_file_name"]
CASCADE_SUMMARY_FILE_NAME = FILENAMES["cascade_defaults_summary_file_name"]
CASCADE_METADATA_FILE_NAME = FILENAMES["cascade_defaults_metadata_file_name"]

CASCADE_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_defaults_results_file_name_csv"]
CASCADE_RESULTS_FILE_NAME_PICKLE = FILENAMES["cascade_defaults_results_file_name_pickle"]

CASCADE_REFERENCE_PROFILE_FILE_NAME = FILENAMES["cascade_defaults_reference_profile_file_name"]

GOLD_CASCADE_LEDGER_FILE_NAME = FILENAMES["gold_cascade_defaults_ledger_file_name"]

STAGE1_CFG = GOLD_CFG["stage1"]
STAGE2_CFG = GOLD_CFG["stage2"]
STAGE3_CFG = GOLD_CFG["stage3"]

RANDOM_SEED = int(GOLD_CFG["random_seed"])

STAGE1_ESTIMATOR_COUNT = int(STAGE1_CFG["estimator_count"])
STAGE1_THRESHOLD_PERCENTILE = float(STAGE1_CFG["threshold_percentile"])

# Fixed-only notebook:
STAGE2_SELECTION_MODE = "fixed"
STAGE2_RANDOM_STATE = int(STAGE2_CFG.get("random_state", RANDOM_SEED))
STAGE2_FIXED_PARAMS = dict(STAGE2_CFG["fixed"]["params"])
STAGE2_FIXED_THRESHOLD_PERCENTILE = float(STAGE2_CFG["fixed"]["threshold_percentile"])

STAGE3_MIN_PRIMARY_SENSOR_HITS = int(STAGE3_CFG["min_primary_sensor_hits"])
STAGE3_MIN_SECONDARY_SENSOR_HITS = int(STAGE3_CFG["min_secondary_sensor_hits"])
STAGE3_ROLLING_WINDOW_SIZE = int(STAGE3_CFG["rolling_window_size"])
STAGE3_MINIMUM_FLAGS_IN_WINDOW = int(STAGE3_CFG["minimum_flags_in_window"])

GOLD_CASCADE_LEDGER_FILE_NAME = FILENAMES["gold_cascade_defaults_ledger_file_name"]

set_wandb_dir_from_config(CONFIG)


GOLD_PREPROCESSED_DATA_PATH = Path(PATHS["gold_preprocessed_data_path"])
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(PATHS["gold_preprocessed_scaled_data_path"])

GOLD_TRAIN_DATA_PATH = Path(PATHS["gold_train_data_path"])
GOLD_TEST_DATA_PATH = Path(PATHS["gold_test_data_path"])
GOLD_FIT_DATA_PATH = Path(PATHS["gold_fit_data_path"])
GOLD_ARTIFACTS_PATH = Path(PATHS["gold_artifacts_dir"])

STAGE1_FEATURES_PATH = Path(PATHS["stage1_features_path"])
STAGE2_FEATURES_PATH = Path(PATHS["stage2_features_path"])
STAGE3_PRIMARY_PATH = Path(PATHS["stage3_primary_path"])
STAGE3_SECONDARY_PATH = Path(PATHS["stage3_secondary_path"])


MODELS_PATH = Path(PATHS["models_root"])

STAGE1_MODELS_PATH = Path(PATHS["cascade_defaults_stage1_models_path"])
STAGE1_MODEL_ARTIFACT_PATH = Path(PATHS["cascade_defaults_stage1_model_artifact_path"])

STAGE2_MODELS_PATH = Path(PATHS["cascade_defaults_stage2_models_path"])
STAGE2_MODEL_ARTIFACT_PATH = Path(PATHS["cascade_defaults_stage2_model_artifact_path"])

CASCADE_RESULTS_PATH_CSV = Path(PATHS["cascade_defaults_results_path_csv"])
CASCADE_RESULTS_PATH_PICKLE = Path(PATHS["cascade_defaults_results_path_pickle"])

CASCADE_THRESHOLDS_PATH = Path(PATHS["cascade_defaults_thresholds_path"])
CASCADE_SUMMARY_PATH = Path(PATHS["cascade_defaults_summary_path"])
CASCADE_METADATA_PATH = Path(PATHS["cascade_defaults_metadata_path"])

CASCADE_REFERENCE_PROFILE_PATH = Path(PATHS["cascade_defaults_reference_profile_path"])

# Logs
LOGS_PATH = Path(PATHS["logs_root"])

# Truths
TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])

# Path Failsafes

GOLD_PREPROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PREPROCESSED_SCALED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TRAIN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)



----

## Start Logging for the Cascade Modeling Stage

Before I load the modeling inputs, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, traceability, and basic pipeline discipline. If I need to check what inputs were used or where a step failed, the log gives me a cleaner record than notebook output alone.

In [ ]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_cascade_defaults.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)


2026-04-05 05:29:54,559 | INFO | capstone.gold | Gold Modeling stage starting
2026-04-05 05:29:54,561 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-04-05 05:29:54,562 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-04-05 05:29:54,564 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-04-05 05:29:54,565 | INFO | capstone.gold | Project Notebooks Path Loaded: /workspace/notebooks
2026-04-05 05:29:54,567 | INFO | capstone.gold | Project Truths Path Loaded: /workspace/artifacts/truths
2026-04-05 05:29:54,568 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-04-05 05:29:54,570 | INFO | capstone.gold | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-04-05 05:29:54,571 | INFO | capstone.gold | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/train
2026-04-05 05:29:54,572 | INFO | capstone.gold | Previous Layer (Silver) Testing Path Loaded: /workspace/data/s

----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the cascade modeling stage.

I am using this mainly for run tracking and artifact registration. It helps document the cascade configuration, inputs, and saved outputs, but it does not change the underlying cascade logic itself.

In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_cascade",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "stage1_threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_fixed_threshold_percentile": STAGE2_FIXED_THRESHOLD_PERCENTILE,
        "stage3_min_primary_sensor_hits": STAGE3_MIN_PRIMARY_SENSOR_HITS,
        "stage3_min_secondary_sensor_hits": STAGE3_MIN_SECONDARY_SENSOR_HITS,
        "stage3_rolling_window_size": STAGE3_ROLLING_WINDOW_SIZE,
        "stage3_minimum_flags_in_window": STAGE3_MINIMUM_FLAGS_IN_WINDOW,
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


2026-04-05 05:29:56,316 | INFO | capstone.gold | W&B initialized: gold__001


----

## Initialize the Cascade Ledger

Here I create the ledger that tracks the main steps taken during the cascade notebook.

I treat the ledger as a structured record of the run. It gives me a cleaner summary of the workflow than relying only on printed notebook output, especially when I need to review or compare cascade runs later.

In [ ]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-04-05 05:30:00,246 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:00.246079+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-04-05T05:30:00.246079+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

----

## Load the Gold Modeling Inputs and Resolve the Parent Truth

This is the point where I load the Gold preprocessing outputs that the cascade model depends on.

In this step I am:
- loading the scaled Gold dataframe
- resolving the parent Gold truth record
- confirming the dataset identity
- updating artifact paths from the truth record when available
- loading the Gold fit, train, and test dataframes
- loading the Stage 1, Stage 2, and Stage 3 feature artifacts

This matters because the cascade notebook should inherit the prepared Gold artifacts rather than rebuilding those inputs on its own.

In [ ]:
logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_SCALED_DATA_PATH)
gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

GOLD_DATASET_NAME = (
    gold_preprocessed_scaled_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
GOLD_DATASET_NAME = GOLD_DATASET_NAME[GOLD_DATASET_NAME != ""]

if len(GOLD_DATASET_NAME) == 0:
    raise ValueError("Gold cascade input dataframe is missing usable meta__dataset values.")

GOLD_DATASET_NAME = str(GOLD_DATASET_NAME.iloc[0]).strip()

gold_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_preprocessed_scaled_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="gold",
    dataset_name=GOLD_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(gold_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(gold_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(gold_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

GOLD_PREPROCESSED_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_preprocessed_path", str(GOLD_PREPROCESSED_DATA_PATH)))
GOLD_FIT_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_fit_path", str(GOLD_FIT_DATA_PATH)))
GOLD_TEST_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_test_path", str(GOLD_TEST_DATA_PATH)))
GOLD_TRAIN_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_train_path", str(GOLD_TRAIN_DATA_PATH)))
STAGE1_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage1_features_path", str(STAGE1_FEATURES_PATH)))
STAGE2_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage2_features_path", str(STAGE2_FEATURES_PATH)))
STAGE3_PRIMARY_PATH = Path(gold_truth_artifact_paths.get("stage3_primary_path", str(STAGE3_PRIMARY_PATH)))
STAGE3_SECONDARY_PATH = Path(gold_truth_artifact_paths.get("stage3_secondary_path", str(STAGE3_SECONDARY_PATH)))

logger.info("Resolved Gold cascade dataset name from Gold truth: %s", DATASET_NAME)
logger.info("Resolved Gold truth path: %s", GOLD_TRUTH_PATH)

print("Gold cascade dataset name from parent truth:", DATASET_NAME)
print("Gold cascade parent truth hash:", GOLD_PARENT_TRUTH_HASH)

logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_DATA_PATH)
gold_preprocessed_dataframe = load_data(GOLD_PREPROCESSED_DATA_PATH)

logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Gold test parquet: %s", GOLD_TEST_DATA_PATH)
gold_test_dataframe = load_data(GOLD_TEST_DATA_PATH)

logger.info("Loading Gold train parquet: %s", GOLD_TRAIN_DATA_PATH)
gold_train_dataframe = load_data(GOLD_TRAIN_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

logger.info("Loading Stage 2 features: %s", STAGE2_FEATURES_PATH)
stage2_feature_columns = load_json(STAGE2_FEATURES_PATH)

logger.info("Loading Stage 3 primary rule sensors: %s", STAGE3_PRIMARY_PATH)
stage3_primary_rule_sensors = load_json(STAGE3_PRIMARY_PATH)

logger.info("Loading Stage 3 secondary rule sensors: %s", STAGE3_SECONDARY_PATH)
stage3_secondary_rule_sensors = load_json(STAGE3_SECONDARY_PATH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold scaled parquet, loaded Gold truth, substituted truth-linked artifact paths, then loaded cascade inputs.",
    data={
        "gold_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_preprocessed_path": str(GOLD_PREPROCESSED_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "gold_train_path": str(GOLD_TRAIN_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_preprocessed_shape": list(gold_preprocessed_dataframe.shape),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "gold_test_shape": list(gold_test_dataframe.shape),
        "gold_train_shape": list(gold_train_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

gold_test_dataframe.head(3)

2026-04-05 05:30:00,541 | INFO | capstone.gold | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-04-05 05:30:00,545 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-04-05 05:30:01,466 | INFO | capstone.gold | Resolved Gold cascade dataset name from Gold truth: pump
2026-04-05 05:30:01,469 | INFO | capstone.gold | Resolved Gold truth path: /workspace/artifacts/truths/gold/pump__gold__truth__d6c00152ed2f56139b4292f313a83aace2cfed3514a960c466053606a9f0be4e.json
2026-04-05 05:30:01,473 | INFO | capstone.gold | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet
2026-04-05 05:30:01,476 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet


Gold cascade dataset name from parent truth: pump
Gold cascade parent truth hash: d6c00152ed2f56139b4292f313a83aace2cfed3514a960c466053606a9f0be4e


2026-04-05 05:30:02,191 | INFO | capstone.gold | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-04-05 05:30:02,193 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-04-05 05:30:02,638 | INFO | capstone.gold | Loading Gold test parquet: /workspace/data/gold/pump__gold__test.parquet
2026-04-05 05:30:02,641 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__test.parquet
2026-04-05 05:30:02,974 | INFO | capstone.gold | Loading Gold train parquet: /workspace/data/gold/pump__gold__train.parquet
2026-04-05 05:30:02,976 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__train.parquet
2026-04-05 05:30:03,418 | INFO | capstone.gold | Loading Stage 1 features: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-04-05 05:30:03,425 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__stage1_features.j

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__row_id,meta__is_train_flag
0,asset__001,pump,5,pump:asset__001:run__001:136431,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,9701747606095593835,run__001,sensor.csv,136431,test,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-07-04 17:51:00+00:00,136431,136431,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-50.814252,-2.274511,-5.888888,-1.236360,-50.126917,-9.021832,-6.255302,-6.828505,-6.049950,-11.333579,-6.540518,-5.119545,-5.997311,-0.902079,0.043854,0.025437,0.189619,0.158404,-0.098290,0.006694,0.050789,0.674963,3.116164,0.436222,0.237276,0.492741,0.629638,0.191590,-0.772389,0.780644,0.234333,0.703432,-0.216653,0.677263,1.453255,0.406195,0.319102,-1.437501,8.590904,-0.109091,-0.863635,20.823540,6.666667,-1.181816,-1.136364,-1.035715,-1.26087,-0.941635,-2.166668,-4.21168,2018-07-04 17:51:00,NORMAL,136431,False
1,asset__001,pump,5,pump:asset__001:run__001:136432,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,4242984989007806471,run__001,sensor.csv,136432,test,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-07-04 17:52:00+00:00,136432,136432,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-50.814253,-2.098039,-5.888888,-1.272725,-49.945099,-9.079268,-6.361687,-6.928511,-6.049950,-11.333579,-6.566318,-5.119545,-5.997311,-0.902079,0.089622,-0.077443,0.357106,0.361669,-0.017151,0.009255,0.056949,0.698316,3.095068,0.416996,0.250370,0.493630,0.425490,0.269709,-0.708270,1.090321,0.975476,0.786631,-0.172121,0.590545,1.320089,0.404733,-0.098002,-1.468751,8.363630,0.000000,-0.909090,21.058842,7.625000,-1.181816,-1.227272,-1.035714,-1.26087,-0.941635,-2.166668,-4.21168,2018-07-04 17:52:00,NORMAL,136432,False
2,asset__001,pump,5,pump:asset__001:run__001:136433,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,11874558860056305371,run__001,sensor.csv,136433,test,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-07-04 17:53:00+00:00,136433,136433,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-50.814253,-1.941177,-5.888888,-1.290906,-50.013281,-9.088074,-6.361687,-6.871359,-6.033293,-11.500207,-6.563817,-5.119545,-5.997311,-0.902079,0.013262,-0.098342,-0.066365,-0.005373,-0.042199,0.069460,0.026548,0.665235,3.039044,0.376989,0.231377,0.507400,0.502814,0.268206,-0.661707,0.677419,0.858314,0.791715,0.050019,0.509348,1.203625,0.393889,0.376575,-1.468751,7.954540,0.072728,-0.909090,20.705878,8.375000,-1.181816,-1.227273,-1.035714,-1.26087,-0.941635,-2.166668,-4.21168,2018-07-04 17:53:00,NORMAL,136433,False


----

In [ ]:
gold_preprocessed_scaled_dataframe = ensure_stable_row_id(
    gold_preprocessed_scaled_dataframe,
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="validate_cascade_row_tracking",
    message="Validated stable row identity on Gold cascade modeling input dataframe.",
    data={
        "row_id_column": "meta__row_id",
        "row_count": int(len(gold_preprocessed_scaled_dataframe)),
        "row_id_unique": bool(gold_preprocessed_scaled_dataframe["meta__row_id"].is_unique),
    },
    logger=logger,
)

2026-04-05 05:30:04,258 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:04.258160+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'validate_cascade_row_tracking', 'message': 'Validated stable row identity on Gold cascade modeling input dataframe.', 'why': None, 'consequence': None, 'data': {'row_id_column': 'meta__row_id', 'row_count': 220320, 'row_id_unique': True}}


{'ts_utc': '2026-04-05T05:30:04.258160+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'validate_cascade_row_tracking',
 'message': 'Validated stable row identity on Gold cascade modeling input dataframe.',
 'why': None,
 'consequence': None,
 'data': {'row_id_column': 'meta__row_id',
  'row_count': 220320,
  'row_id_unique': True}}

----

## Rebuild the Train and Test Masks

Before fitting or evaluating the cascade, I recover the train/test split that was already stamped during Gold preprocessing.

The key point here is that this notebook is **not** creating a new split. It is reusing the split created upstream so the baseline notebook, this cascade notebook, and the comparison notebook all work from the same row partition.

In [ ]:
# Masks
if "meta__is_train_flag" not in gold_preprocessed_scaled_dataframe.columns:
    raise ValueError("meta__is_train_flag missing from gold_preprocessed_scaled_dataframe. "
                     "Gold preprocessing must stamp it before saving.")

train_mask = gold_preprocessed_scaled_dataframe["meta__is_train_flag"].astype(bool)
test_mask = ~train_mask

logger.info("Split counts: all=%d train=%d test=%d", len(train_mask), int(train_mask.sum()), int(test_mask.sum()))

2026-04-05 05:30:04,560 | INFO | capstone.gold | Split counts: all=220320 train=136431 test=83889


### Ask

Why does the cascade notebook reuse the saved split instead of building a fresh one here?

### Answer

I want the evaluation setup to stay consistent across the Gold modeling notebooks.

If the baseline and cascade notebooks each made their own split, then the results would be harder to compare fairly. By reusing the saved Gold split, I know both modeling approaches are being judged against the same training and test boundary.

So this step is mainly about consistency and comparability.

----

## Define the Stage 3 Reference Profile Logic

Before Stage 3 can confirm alerts, I need a reference profile that describes what normal feature behavior looks like.

This helper function builds that profile from the Gold fit dataframe by summarizing each selected feature using values like:
- median
- mean
- standard deviation
- lower bound
- upper bound

The main idea is that Stage 3 should not just look for model flags. It should also compare suspicious rows against a stored picture of normal behavior.

In [ ]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

## Build the Stage 3 Reference Profile

Here I create the actual reference profile used by Stage 3 confirmation logic.

The feature set for this profile combines:
- the broader Stage 1 modeling features
- the Stage 3 primary rule sensors
- the Stage 3 secondary rule sensors

That gives Stage 3 a normal-behavior reference it can use when checking for profile breaches and rule evidence.

In [ ]:
reference_profile_features = list(dict.fromkeys(
    stage1_feature_columns + stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

reference_profile = build_reference_profile(
    gold_fit_dataframe,
    feature_columns=reference_profile_features,
)

ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built fit-period reference profile for Stage 3 confirmation logic.",
    data={
        "training_rows": int(len(gold_fit_dataframe)),
        "reference_feature_count": int(len(reference_profile_features)),
    },
    logger=logger,
)

reference_profile.head(10)

2026-04-05 05:30:05,555 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:05.555623+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'build_reference_profile', 'message': 'Built fit-period reference profile for Stage 3 confirmation logic.', 'why': None, 'consequence': None, 'data': {'training_rows': 122065, 'reference_feature_count': 50}}


,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,0.0,-1.191161,6.117528,-6.418646,1.302319
1,sensor_01,0.0,0.014146,0.838560,-1.313726,1.431370
2,sensor_02,0.0,-0.045551,0.814930,-1.259258,1.055557
3,sensor_03,0.0,-0.003276,0.729044,-1.145452,1.145452
4,sensor_04,0.0,-0.874445,5.304649,-3.770447,1.181813
5,sensor_05,0.0,-0.036448,1.103071,-1.376761,1.517739
6,sensor_06,0.0,-0.212238,1.983681,-1.425542,1.808494
7,sensor_07,0.0,-0.275468,1.148624,-1.085688,1.028576
8,sensor_08,0.0,0.042985,1.223742,-0.949980,1.283344
9,sensor_09,0.0,-0.434937,2.478143,-1.700060,0.366665


----

## Prepare the Feature Matrices and Evaluation Labels

Now I build the actual feature matrices that the cascade will use.

This includes:
- Stage 1 fit features from the Gold fit dataframe
- Stage 2 fit features from the Gold fit dataframe
- Stage 1 full-dataset scoring features
- Stage 2 full-dataset scoring features
- test labels, when `anomaly_flag` is available

A detail that matters here is that the model fitting happens on the **normal-only Gold fit subset**, while scoring happens across the **full scaled Gold dataset**.

In [ ]:
# Fit features from normal-only fit parquet
stage1_train_fit_features = gold_fit_dataframe[stage1_feature_columns].values
stage2_train_fit_features = gold_fit_dataframe[stage2_feature_columns].values

# Score features from the full scaled dataset (ALL rows)
stage1_all_features = gold_preprocessed_scaled_dataframe[stage1_feature_columns].values
stage2_all_features = gold_preprocessed_scaled_dataframe[stage2_feature_columns].values


test_labels = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe
        .loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .values
    )

### Ask

Why am I fitting on the Gold fit subset but scoring the full scaled dataset?

### Answer

Because those steps are doing different jobs.

The fit subset gives the cascade its reference view of normal behavior. The full scaled dataset is what I want to score afterward so I can see where alerts appear across all rows.

So the overall logic is:
- fit the Stage 1 and Stage 2 models on the saved normal training subset
- score every row in the scaled Gold dataset
- evaluate the final cascade outcome on the held-out test rows

----

## Define Shared Scoring, Thresholding, and Evaluation Helpers

These helper functions break the cascade workflow into reusable pieces:
- computing anomaly scores from an Isolation Forest
- choosing a threshold by percentile
- evaluating predicted alerts against available labels

I like separating these pieces because it keeps the notebook easier to read and also makes the relationship between the baseline and cascade logic more obvious.

In [ ]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [ ]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [ ]:

def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

----

## Run Stage 1: Broad Isolation Forest Screening

This is the first model stage of the cascade.

Stage 1 uses the broader feature set to act as an initial screen. The goal here is to cast a wider net and identify rows that look suspicious enough to move forward in the cascade.

In this step I:
- fit the Stage 1 Isolation Forest on the Gold fit subset
- score the fit subset and the full scaled dataset
- choose the Stage 1 threshold from the training-score distribution
- create the Stage 1 alert flags

This stage is intentionally broad. It is supposed to surface candidate alerts, not be the final decision by itself.

In [ ]:
stage1_model = IsolationForest(
    n_estimators=STAGE1_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage1_model.fit(stage1_train_fit_features)

stage1_train_scores = compute_anomaly_scores_isolation_forest(
    stage1_model, 
    stage1_train_fit_features
)

stage1_all_scores = compute_anomaly_scores_isolation_forest(
    stage1_model, 
    stage1_all_features
)

stage1_threshold = choose_threshold_by_percentile(
    stage1_train_scores, 
    STAGE1_THRESHOLD_PERCENTILE
)

stage1_flags = (stage1_all_scores >= stage1_threshold).astype(int)

stage1_summary = {
    "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "threshold": float(stage1_threshold),
    "alert_count_all_rows": int(stage1_flags.sum()),
    "alert_count_test_rows": int(stage1_flags[test_mask.values].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage1",
    message="Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.",
    data={
        "estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "threshold": float(stage1_threshold),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_all_rows": int(stage1_summary["alert_count_all_rows"]),
        "alert_count_test_rows": int(stage1_summary["alert_count_test_rows"]),
    },
    logger=logger,
)

2026-04-05 05:30:09,803 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:09.803925+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage1', 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 95.0, 'threshold': 0.5337445171501773, 'feature_count': 50, 'alert_count_all_rows': 22742, 'alert_count_test_rows': 2288}}


{'ts_utc': '2026-04-05T05:30:09.803925+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'run_cascade_stage1',
 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.',
 'why': None,
 'consequence': None,
 'data': {'estimator_count': 200,
  'threshold_percentile': 95.0,
  'threshold': 0.5337445171501773,
  'feature_count': 50,
  'alert_count_all_rows': 22742,
  'alert_count_test_rows': 2288}}

### Ask

What is Stage 1 really supposed to do in the cascade?

### Answer

Stage 1 is the broad screening step.

Its job is to catch rows that look suspicious enough to deserve a second look. I do not expect Stage 1 alone to be highly selective. In fact, it is normal for Stage 1 to raise more alerts than the later stages, because the later stages are there to narrow and confirm those alerts.

So when I review Stage 1, I mainly care that it is acting as a reasonable first filter rather than as the final answer.

----

In [ ]:
stage1_input_df = build_stage_scoring_frame(
    dataframe=gold_preprocessed_scaled_dataframe,
    feature_columns=stage1_feature_columns,
    mask=None,
    row_id_column="meta__row_id",
)

stage1_results_df = score_isolation_forest_stage(
    stage_dataframe=stage1_input_df,
    model=stage1_model,
    feature_columns=stage1_feature_columns,
    stage_name="stage1",
    row_id_column="meta__row_id",
)

cascade_results = merge_stage_results_back(
    master_dataframe=gold_preprocessed_scaled_dataframe.copy(),
    stage_results_dataframe=stage1_results_df,
    stage_name="stage1",
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="score_stage1_with_row_tracking",
    message="Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.",
    data={
        "stage1_row_count": int(len(stage1_results_df)),
        "stage1_flag_count": int(stage1_results_df["stage1_flag"].sum()),
    },
    logger=logger,
)

/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
2026-04-05 05:30:14,534 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:14.534009+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'score_stage1_with_row_tracking', 'message': 'Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.', 'why': None, 'consequence': None, 'data': {'stage1_row_count': 220320, 'stage1_flag_count': 35615}}


{'ts_utc': '2026-04-05T05:30:14.534009+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'score_stage1_with_row_tracking',
 'message': 'Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.',
 'why': None,
 'consequence': None,
 'data': {'stage1_row_count': 220320, 'stage1_flag_count': 35615}}

---

## Run Stage 2: Narrow Isolation Forest Confirmation

This is the second model stage of the cascade.

Stage 2 uses the narrower Stage 2 feature set and the fixed parameter branch from the cascade configuration. The goal here is to confirm a smaller subset of the Stage 1 alerts rather than scoring rows completely independently.

That is why the final Stage 2 flag is created by combining:
- the raw Stage 2 alert
- the Stage 1 alert

So a row has to pass through both model stages before it can move into Stage 3 confirmation logic.

In [ ]:
stage2_model = IsolationForest(
    random_state=STAGE2_RANDOM_STATE,
    n_jobs=-1,
    **STAGE2_FIXED_PARAMS,
)

stage2_model.fit(stage2_train_fit_features)

stage2_train_scores = compute_anomaly_scores_isolation_forest(
    stage2_model,
    stage2_train_fit_features,
)

stage2_all_scores = compute_anomaly_scores_isolation_forest(
    stage2_model,
    stage2_all_features,
)

stage2_threshold = choose_threshold_by_percentile(
    stage2_train_scores,
    STAGE2_FIXED_THRESHOLD_PERCENTILE,
)

stage2_raw_flags = (stage2_all_scores >= stage2_threshold).astype(int)
stage2_flags = ((stage1_flags == 1) & (stage2_raw_flags == 1)).astype(int)

stage2_selected_threshold_percentile = float(STAGE2_FIXED_THRESHOLD_PERCENTILE)
stage2_best_params = dict(STAGE2_FIXED_PARAMS)

stage2_summary = {
    "selection_mode": "fixed",
    "selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "threshold": float(stage2_threshold),
    "best_params": stage2_best_params,
    "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
    "raw_alert_count_test_rows": int(stage2_raw_flags[test_mask.values].sum()),
    "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
    "stage2_confirmed_count_test_rows": int(stage2_flags[test_mask.values].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage2_fixed",
    message="Ran Stage 2 fixed narrow Isolation Forest using the fixed branch from the cascade config.",
    data={
        "selection_mode": "fixed",
        "best_params": stage2_best_params,
        "selected_threshold_percentile": float(stage2_selected_threshold_percentile),
        "threshold": float(stage2_threshold),
        "feature_count": int(len(stage2_feature_columns)),
        "raw_alert_count_all_rows": int(stage2_summary["raw_alert_count_all_rows"]),
        "raw_alert_count_test_rows": int(stage2_summary["raw_alert_count_test_rows"]),
        "stage2_confirmed_count_all_rows": int(stage2_summary["stage2_confirmed_count_all_rows"]),
        "stage2_confirmed_count_test_rows": int(stage2_summary["stage2_confirmed_count_test_rows"]),
    },
    logger=logger,
)

2026-04-05 05:30:18,275 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:18.275395+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage2_fixed', 'message': 'Ran Stage 2 fixed narrow Isolation Forest using the fixed branch from the cascade config.', 'why': None, 'consequence': None, 'data': {'selection_mode': 'fixed', 'best_params': {'n_estimators': 300, 'max_samples': 'auto', 'contamination': 'auto', 'max_features': 1.0, 'bootstrap': False}, 'selected_threshold_percentile': 98.5, 'threshold': 0.5862519236715605, 'feature_count': 12, 'raw_alert_count_all_rows': 21605, 'raw_alert_count_test_rows': 6065, 'stage2_confirmed_count_all_rows': 16545, 'stage2_confirmed_count_test_rows': 1176}}


{'ts_utc': '2026-04-05T05:30:18.275395+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'run_cascade_stage2_fixed',
 'message': 'Ran Stage 2 fixed narrow Isolation Forest using the fixed branch from the cascade config.',
 'why': None,
 'consequence': None,
 'data': {'selection_mode': 'fixed',
  'best_params': {'n_estimators': 300,
   'max_samples': 'auto',
   'contamination': 'auto',
   'max_features': 1.0,
   'bootstrap': False},
  'selected_threshold_percentile': 98.5,
  'threshold': 0.5862519236715605,
  'feature_count': 12,
  'raw_alert_count_all_rows': 21605,
  'raw_alert_count_test_rows': 6065,
  'stage2_confirmed_count_all_rows': 16545,
  'stage2_confirmed_count_test_rows': 1176}}

### Ask

Why does Stage 2 use both its own raw flag and the Stage 1 flag?

### Answer

Because Stage 2 is acting as a confirmation layer, not as a separate standalone model result.

The raw Stage 2 score tells me what the narrower model thinks. But the actual Stage 2 cascade flag only turns on when the row was already flagged by Stage 1 and then also confirmed by Stage 2.

That means Stage 2 is narrowing the Stage 1 alert set instead of replacing it.

----

In [ ]:
stage2_candidate_mask = cascade_results["stage1_flag"].fillna(0).astype(int) == 1

stage2_input_df = build_stage_scoring_frame(
    dataframe=cascade_results,
    feature_columns=stage2_feature_columns,
    mask=stage2_candidate_mask,
    row_id_column="meta__row_id",
)

stage2_results_df = score_isolation_forest_stage(
    stage_dataframe=stage2_input_df,
    model=stage2_model,
    feature_columns=stage2_feature_columns,
    stage_name="stage2",
    row_id_column="meta__row_id",
)

# Rename raw helper outputs so they do not overwrite your threshold-based final stage2 flag logic
stage2_results_df = stage2_results_df.rename(
    columns={
        "stage2_score": "stage2_model_score",
        "stage2_decision": "stage2_model_decision",
        "stage2_pred": "stage2_model_pred",
        "stage2_flag": "stage2_model_flag",
    }
)


cascade_results = cascade_results.merge(
    stage2_results_df[
        [
            "meta__row_id",
            "stage2_model_score",
            "stage2_model_decision",
            "stage2_model_pred",
            "stage2_model_flag",
        ]
    ],
    on="meta__row_id",
    how="left",
)

cascade_results["stage2_score"] = np.nan
cascade_results.loc[stage2_candidate_mask, "stage2_score"] = stage2_all_scores[stage2_candidate_mask.values]

cascade_results["stage2_raw_flag"] = 0
cascade_results.loc[stage2_candidate_mask, "stage2_raw_flag"] = stage2_raw_flags[stage2_candidate_mask.values]

cascade_results["stage2_flag"] = 0
cascade_results.loc[stage2_candidate_mask, "stage2_flag"] = stage2_flags[stage2_candidate_mask.values]

cascade_results["stage2_raw_flag"] = cascade_results["stage2_raw_flag"].fillna(0).astype(int)
cascade_results["stage2_flag"] = cascade_results["stage2_flag"].fillna(0).astype(int)

ledger.add(
    kind="step",
    step="score_stage2_with_row_tracking",
    message="Scored Stage 2 on Stage 1 candidate rows and merged row-level outputs back using stable row identity.",
    data={
        "stage2_candidate_row_count": int(stage2_candidate_mask.sum()),
        "stage2_scored_row_count": int(len(stage2_results_df)),
        "stage2_raw_flag_count": int(cascade_results["stage2_raw_flag"].sum()),
        "stage2_flag_count": int(cascade_results["stage2_flag"].sum()),
    },
    logger=logger,
)

/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
2026-04-05 05:30:19,658 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:19.658536+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'score_stage2_with_row_tracking', 'message': 'Scored Stage 2 on Stage 1 candidate rows and merged row-level outputs back using stable row identity.', 'why': None, 'consequence': None, 'data': {'stage2_candidate_row_count': 35615, 'stage2_scored_row_count': 35615, 

{'ts_utc': '2026-04-05T05:30:19.658536+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'score_stage2_with_row_tracking',
 'message': 'Scored Stage 2 on Stage 1 candidate rows and merged row-level outputs back using stable row identity.',
 'why': None,
 'consequence': None,
 'data': {'stage2_candidate_row_count': 35615,
  'stage2_scored_row_count': 35615,
  'stage2_raw_flag_count': 18301,
  'stage2_flag_count': 16545}}

----

## Continue the Cascade Results Dataframe from Row-Tracked Stage Outputs

At this point the working `cascade_results` dataframe already contains the Stage 1 row-level outputs merged by stable row identity, and Stage 2 row-level outputs are now merged back in for the Stage 1 candidate rows.

So I do not rebuild the results dataframe from scratch here. Instead, I continue using the row-tracked `cascade_results` dataframe as the single source of truth for Stage 3 rule logic.

----

## Validate That the Stage 3 Rule Sensors Exist

Before I run Stage 3 confirmation logic, I want to verify that the required primary and secondary rule sensors are actually present in the scored dataframe.

This is a quick trust check. If those sensors are missing, then the Stage 3 confirmation logic would be working from an incomplete rule set, which would weaken the final cascade decision.

In [ ]:
# --- Stage 3 sanity check: ensure rule sensors exist in scored dataframe
missing_primary = [column for column in stage3_primary_rule_sensors if column not in cascade_results.columns]
missing_secondary = [column for column in stage3_secondary_rule_sensors if column not in cascade_results.columns]

logger.info("Stage3 missing sensors: primary=%d secondary=%d", len(missing_primary), len(missing_secondary))

if missing_primary:
    logger.warning("Missing Stage3 PRIMARY sensors (showing up to 20): %s", missing_primary[:20])
if missing_secondary:
    logger.warning("Missing Stage3 SECONDARY sensors (showing up to 20): %s", missing_secondary[:20])


2026-04-05 05:30:19,976 | INFO | capstone.gold | Stage3 missing sensors: primary=0 secondary=0


----

## Compute the Primary Profile Breach Count

Here I apply the primary breach logic to the cascade results dataframe.

This creates a row-level count of how many Stage 3 primary sensors are outside the expected normal profile bounds. That count becomes one of the strongest pieces of Stage 3 evidence.

In [ ]:

def compute_primary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name")[["lower_bound", "upper_bound"]]

    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns or feature_name not in reference_lookup.index:
            continue

        lower = reference_lookup.loc[feature_name, "lower_bound"]
        upper = reference_lookup.loc[feature_name, "upper_bound"]

        breach_flag = ((dataframe[feature_name] < lower) | (dataframe[feature_name] > upper)).astype(int)
        breach_counts = breach_counts + breach_flag

    breach_counts.name = "stage3_profile_breach_count"
    return breach_counts





## Define the Secondary Corroboration Logic

This helper function does a similar job for the secondary rule sensors.

The difference is that these secondary sensors are used more as corroboration than as the main profile breach signal. In other words, they help support the alert rather than carrying the same priority as the primary sensor group.

In [ ]:
cascade_results["stage3_profile_breach_count"] = compute_primary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_primary_rule_sensors,
)

----

## Compute the Secondary Breach Count

Here I apply the secondary breach logic to the cascade results dataframe.

This creates the row-level count of secondary rule sensors that move outside their reference profile bounds. Later this gets converted into a corroboration flag that contributes to the Stage 3 evidence count.

In [ ]:
def compute_secondary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]

        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound) |
            (dataframe[feature_name] > upper_bound)
        ).astype(int)

        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = "stage3_secondary_breach_count"
    return breach_counts

In [ ]:
cascade_results["stage3_secondary_breach_count"] = compute_secondary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_secondary_rule_sensors,
)

----

## Define the Persistence Logic

Not every abnormal point matters equally. Some are isolated spikes, while others persist across nearby rows.

This helper function checks for persistence by looking at recent Stage 2 flags inside a rolling window. If enough flags appear within that window, the row receives a persistence flag.

In [ ]:
def compute_persistence_flag(
    source_flags: pd.Series,
    *,
    rolling_window_size: int = 3,
    minimum_flags_in_window: int = 2,
) -> pd.Series:
    persistence_flag = (
        source_flags
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
        .ge(minimum_flags_in_window)
        .astype(int)
    )

    persistence_flag.name = "stage3_persistence_flag"
    return persistence_flag

## Compute the Persistence Flag

Here I apply the persistence logic to the Stage 2 flags.

The goal is to capture short-run continuity. A row that belongs to a cluster of nearby Stage 2 flags has stronger evidence than a single one-off hit by itself.

In [ ]:
cascade_results["stage3_persistence_flag"] = compute_persistence_flag(
    cascade_results["stage2_flag"],
    rolling_window_size=STAGE3_ROLLING_WINDOW_SIZE,
    minimum_flags_in_window=STAGE3_MINIMUM_FLAGS_IN_WINDOW,
)


----

## Define the Drift Logic

This helper function checks for rolling drift behavior in the Stage 3 watch features.

The idea is to compare each feature to its recent rolling median and flag rows where the current value drifts far enough away from that recent local behavior. This gives Stage 3 another kind of evidence that is different from simple threshold breaches.

In [ ]:
def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    rolling_window_size: int = 5,
    drift_threshold_multiplier: float = 1.0,
) -> pd.Series:
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()

        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()

        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)

        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    drift_flag = (drift_trigger_counts >= 1).astype(int)
    drift_flag.name = "stage3_drift_flag"
    return drift_flag

----

## Compute the Drift Flag

Here I apply the drift logic to the combined Stage 3 watch feature set.

This produces a row-level drift flag that tells me whether at least one watched feature is moving far enough away from its recent rolling pattern to count as drift evidence.

In [ ]:
stage3_rule_watch_features = list(dict.fromkeys(
    stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

cascade_results["stage3_drift_flag"] = compute_drift_flag(
    cascade_results,
    feature_columns=stage3_rule_watch_features,
    rolling_window_size=5,
    drift_threshold_multiplier=1.0,
)

----

## Build the Final Stage 3 Evidence Flags and Final Cascade Decision

Now I combine the Stage 3 evidence into the final confirmation logic.

This section creates:
- the primary profile breach flag
- the secondary corroboration flag
- the persistence flag
- the drift flag
- the overall Stage 3 evidence count
- the final `cascade_final_flag`

The final cascade decision is intentionally strict. A row must first pass through Stage 1 and Stage 2, and then it must also show enough Stage 3 evidence to be considered a final cascade alert.

In [ ]:
cascade_results["stage3_profile_breach_flag"] = (
    cascade_results["stage3_profile_breach_count"] >= STAGE3_MIN_PRIMARY_SENSOR_HITS
).astype(int)

cascade_results["stage3_corroboration_flag"] = (
    cascade_results["stage3_secondary_breach_count"] >= STAGE3_MIN_SECONDARY_SENSOR_HITS
).astype(int)

cascade_results["stage3_rule_evidence_count"] = (
    cascade_results["stage3_profile_breach_flag"] +
    cascade_results["stage3_persistence_flag"] +
    cascade_results["stage3_drift_flag"] +
    cascade_results["stage3_corroboration_flag"]
)

cascade_results["cascade_final_flag"] = (
    (cascade_results["stage1_flag"] == 1) &
    (cascade_results["stage2_flag"] == 1) &
    (
        (cascade_results["stage3_profile_breach_count"] >= STAGE3_MIN_PRIMARY_SENSOR_HITS) |
        (cascade_results["stage3_rule_evidence_count"] >= 2)
    )
).astype(int)

stage3_summary = {
    "primary_rule_sensor_count": int(len(stage3_primary_rule_sensors)),
    "secondary_rule_sensor_count": int(len(stage3_secondary_rule_sensors)),
    "profile_breach_rows_all": int((cascade_results["stage3_profile_breach_flag"] == 1).sum()),
    "profile_breach_rows_test": int(cascade_results.loc[test_mask, "stage3_profile_breach_flag"].sum()),
    "corroboration_rows_all": int((cascade_results["stage3_corroboration_flag"] == 1).sum()),
    "corroboration_rows_test": int(cascade_results.loc[test_mask, "stage3_corroboration_flag"].sum()),
    "persistence_rows_all": int((cascade_results["stage3_persistence_flag"] == 1).sum()),
    "persistence_rows_test": int(cascade_results.loc[test_mask, "stage3_persistence_flag"].sum()),
    "drift_rows_all": int((cascade_results["stage3_drift_flag"] == 1).sum()),
    "drift_rows_test": int(cascade_results.loc[test_mask, "stage3_drift_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage3_confirmation",
    message="Applied Stage 3 confirmation checks to all rows of the scaled dataset.",
    data=stage3_summary,
    logger=logger,
)

cascade_results.head(5)

2026-04-05 05:30:23,349 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:23.349908+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage3_confirmation', 'message': 'Applied Stage 3 confirmation checks to all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'primary_rule_sensor_count': 5, 'secondary_rule_sensor_count': 7, 'profile_breach_rows_all': 72702, 'profile_breach_rows_test': 49472, 'corroboration_rows_all': 150151, 'corroboration_rows_test': 80276, 'persistence_rows_all': 16535, 'persistence_rows_test': 1165, 'drift_rows_all': 2448, 'drift_rows_test': 1409, 'final_alert_count_all_rows': 16501, 'final_alert_count_test_rows': 1166}}


,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__row_id,meta__is_train_flag,stage1_score,stage1_decision,stage1_pred,stage1_flag,stage2_model_score,stage2_model_decision,stage2_model_pred,stage2_model_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_rule_evidence_count,cascade_final_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,14598431322315673869,run__001,sensor.csv,0,train,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.232560,-0.509807,0.537036,1.090905,0.227271,-0.144184,-0.212771,0.000000,0.633343,-0.133358,-0.966540,0.663285,-0.099631,-0.152932,0.001485,-0.017980,0.259816,0.203007,0.012069,0.001266,0.026039,-0.004667,0.375041,0.470634,0.054448,0.018867,-0.398656,-0.746314,-0.137762,-0.554838,-1.070844,-0.832376,-0.653530,-0.038647,-0.157592,-0.223411,0.170945,-1.156251,-0.909090,0.527273,-0.909090,-0.882353,-0.125000,0.000000,4.000000,0.928571,-0.608697,0.715953,1.900001,0.014598,2018-04-01 00:00:00,NORMAL,0,True,-0.391431,0.108569,1,0,NaN,NaN,NaN,NaN,NaN,0,0,0,1,0,0,0,1,1,0
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,15954729095895098000,run__001,sensor.csv,1,train,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.232560,-0.509807,0.537036,1.090905,0.227271,-0.144184,-0.212771,0.000000,0.633343,-0.133358,-0.966540,0.663285,-0.099631,-0.152932,0.001485,-0.017980,0.259816,0.203007,0.012069,0.001266,0.026039,-0.004667,0.375041,0.470634,0.054448,0.018867,-0.398656,-0.746314,-0.137762,-0.554838,-1.070844,-0.832376,-0.653530,-0.038647,-0.157592,-0.223411,0.170945,-1.156251,-0.909090,0.527273,-0.909090,-0.882353,-0.125000,0.000000,4.000000,0.928571,-0.608697,0.715953,1.900001,0.014598,2018-04-01 00:01:00,NORMAL,1,True,-0.391431,0.108569,1,0,NaN,NaN,NaN,NaN,NaN,0,0,0,1,0,0,0,1,1,0
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,10041703297090838359,run__001,sensor.csv,2,train,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.255821,-0.392159,0.537036,1.127271,0.670453,-0.485083,-0.468102,-0.185694,0.750017,-0.333349,-0.869635,0.742744,0.084552,-0.140848,0.063680,0.024252,-0.023257,-0.051481,0.031807,0.039803,0.034031,0.045869,0.539582,0.559032,0.045512,0.031031,-0.029962,-0.769896,-0.025671,0.380643,-0.866485,-0.748481,-0.586075,-0.053782,-0.147948,-0.214093,0.276319,-1.031250,-0.954545,0.454546,-0.999999,-0.882353,-0.166667,-0.045454,3.954546,0.964285,-0.608696,0.688715,1.833334,0.072992,2018-04-01 00:02:00,NORMAL,2,True,-0.384923,0.115077,1,0,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0
3,asset__001,pump,0,pump:asset__001:run__001:3,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9

In [ ]:

cascade_results = finalize_stage_flag_columns(
    cascade_results,
    stage_names=["stage1", "stage2", "stage3"],
)

if "stage3_confirmed_flag" in cascade_results.columns and "stage3_flag" not in cascade_results.columns:
    cascade_results["stage3_flag"] = cascade_results["stage3_confirmed_flag"].fillna(0).astype(int)

ledger.add(
    kind="step",
    step="finalize_stage_flags",
    message="Finalized sparse cascade stage flags after Stage 3 rule processing.",
    data={
        "stage1_flag_count": int(cascade_results["stage1_flag"].sum()) if "stage1_flag" in cascade_results.columns else 0,
        "stage2_flag_count": int(cascade_results["stage2_flag"].sum()) if "stage2_flag" in cascade_results.columns else 0,
        "stage3_flag_count": int(cascade_results["stage3_flag"].sum()) if "stage3_flag" in cascade_results.columns else 0,
        "cascade_final_flag_count": int(cascade_results["cascade_final_flag"].sum()) if "cascade_final_flag" in cascade_results.columns else 0,
    },
    logger=logger,
)

2026-04-05 05:30:24,143 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:24.143955+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_stage_flags', 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.', 'why': None, 'consequence': None, 'data': {'stage1_flag_count': 35615, 'stage2_flag_count': 16545, 'stage3_flag_count': 0, 'cascade_final_flag_count': 16501}}


{'ts_utc': '2026-04-05T05:30:24.143955+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'finalize_stage_flags',
 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.',
 'why': None,
 'consequence': None,
 'data': {'stage1_flag_count': 35615,
  'stage2_flag_count': 16545,
  'stage3_flag_count': 0,
  'cascade_final_flag_count': 16501}}

### Ask

How should I think about the final Stage 3 rule?

### Answer

Stage 3 is the confirmation layer that tries to reduce weak or isolated model alerts.

In this notebook, a row can become a final cascade alert in one of two main ways:
- it shows enough primary profile breaches on its own
- or it builds enough combined rule evidence across the Stage 3 checks

So Stage 3 is not replacing the model stages. It is acting as a final confirmation layer after the row has already passed through both model filters.

----

## Build the Main Cascade Metrics

Here I summarize the main cascade results.

This includes:
- Stage 1 alert counts
- Stage 2 alert counts
- final cascade alert counts
- test-row alert counts
- evaluation metrics such as precision, recall, and F1 when test labels are available

This is the main performance snapshot for the default cascade notebook.

In [ ]:
cascade_metrics = {
    "model": "3-Stage Cascade",
    "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_all_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "stage1_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage1_flag"].sum()),
    "stage2_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage2_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
}



if test_labels is not None:
    cascade_test_flags = (
        cascade_results
        .loc[test_mask, "cascade_final_flag"]
        .astype(int)
        .values
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels,
        cascade_test_flags,
        average="binary",
        zero_division=0,
    )

    cascade_metrics["precision"] = float(precision)
    cascade_metrics["recall"] = float(recall)
    cascade_metrics["f1"] = float(f1)

cascade_metrics

{'model': '3-Stage Cascade',
 'stage1_alert_count_all_rows': 35615,
 'stage2_alert_count_all_rows': 16545,
 'final_alert_count_all_rows': 16501,
 'stage1_alert_count_test_rows': 6215,
 'stage2_alert_count_test_rows': 1176,
 'final_alert_count_test_rows': 1166,
 'precision': 0.05574614065180103,
 'recall': 0.5508474576271186,
 'f1': 0.10124610591900311}

### Ask

What should I pay attention to in the cascade metrics?

### Answer

The main things I care about are:
- how many rows survive from Stage 1 into Stage 2
- how many rows survive from Stage 2 into the final cascade output
- whether the final alert count drops in a controlled way rather than collapsing too aggressively
- precision, recall, and F1 on the test rows when labels are available

For this notebook, the key question is not just whether the cascade detects anomalies. It is whether the layered filtering helps produce a cleaner alert set than the baseline.

----

In [ ]:
cascade_results = finalize_stage_flag_columns(
    cascade_results,
    stage_names=["stage1", "stage2", "stage3"],
)

if "stage3_confirmed_flag" in cascade_results.columns and "stage3_flag" not in cascade_results.columns:
    cascade_results["stage3_flag"] = cascade_results["stage3_confirmed_flag"].fillna(0).astype(int)

ledger.add(
    kind="step",
    step="finalize_stage_flags",
    message="Finalized sparse cascade stage flags after Stage 3 rule processing.",
    data={
        "stage1_flag_count": int(cascade_results["stage1_flag"].sum()) if "stage1_flag" in cascade_results.columns else 0,
        "stage2_flag_count": int(cascade_results["stage2_flag"].sum()) if "stage2_flag" in cascade_results.columns else 0,
        "stage3_flag_count": int(cascade_results["stage3_flag"].sum()) if "stage3_flag" in cascade_results.columns else 0,
        "cascade_final_flag_count": int(cascade_results["cascade_final_flag"].sum()) if "cascade_final_flag" in cascade_results.columns else 0,
    },
    logger=logger,
)

2026-04-05 05:30:24,817 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:30:24.817806+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_stage_flags', 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.', 'why': None, 'consequence': None, 'data': {'stage1_flag_count': 35615, 'stage2_flag_count': 16545, 'stage3_flag_count': 0, 'cascade_final_flag_count': 16501}}


{'ts_utc': '2026-04-05T05:30:24.817806+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'finalize_stage_flags',
 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.',
 'why': None,
 'consequence': None,
 'data': {'stage1_flag_count': 35615,
  'stage2_flag_count': 16545,
  'stage3_flag_count': 0,
  'cascade_final_flag_count': 16501}}

----

## Build the Cascade Summary, Threshold Records, and Truth Artifact

Now I convert the cascade results into formal pipeline artifacts.

This section does several important things:
- summarizes the cascade thresholds and metrics
- builds the main cascade summary record
- creates the Gold cascade truth record
- stores runtime facts and artifact paths
- links the cascade output back to the parent Gold truth
- prepares the final saved deliverables for downstream comparison

At this point, the notebook output becomes more than just a scored dataframe. It becomes a tracked modeling artifact in the pipeline.

In [ ]:
stage1_alert_count_all_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_all_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_all_rows = int(cascade_results["cascade_final_flag"].sum())

stage1_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage1_flag"].sum())
stage2_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage2_flag"].sum())
final_cascade_alert_count_test_rows = int(cascade_results.loc[test_mask, "cascade_final_flag"].sum())

cascade_thresholds = {
    "cascade_variant": CASCADE_VARIANT,
    "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "stage1_threshold": float(stage1_threshold),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage2_selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "stage2_threshold": float(stage2_threshold),
    "stage2_best_params": stage2_best_params,
}

cascade_summary = {
    "dataset_name": DATASET_NAME,
    "cascade_variant": CASCADE_VARIANT,
    "cascade_metrics": cascade_metrics,
    "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
    "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
    "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
    "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
    "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "result_row_count": int(len(cascade_results)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage2_best_params": stage2_best_params,
}

truth_config_snapshot = (
    TRUTH_CONFIG
    if "TRUTH_CONFIG" in globals()
    else {
        "runtime": {
            "stage": "gold_cascade",
            "dataset": DATASET_NAME,
            "cascade_variant": CASCADE_VARIANT,
            "mode": RUN_MODE if "RUN_MODE" in globals() else None,
            "profile": CONFIG_PROFILE if "CONFIG_PROFILE" in globals() else "default",
        }
    }
)

cascade_truth_layer_name = "gold_cascade"
cascade_process_run_id = (
    GOLD_PROCESS_RUN_ID
    if "GOLD_PROCESS_RUN_ID" in globals()
    else make_process_run_id("gold_cascade_process")
)

cascade_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
    process_run_id=cascade_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "config_snapshot",
    truth_config_snapshot,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "runtime_facts",
    {
        "cascade_variant": CASCADE_VARIANT,
        "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "stage1_threshold": float(stage1_threshold),
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_selected_threshold_percentile": float(stage2_selected_threshold_percentile),
        "stage2_threshold": float(stage2_threshold),
        "stage1_estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "stage2_estimator_count": int(stage2_model.get_params()["n_estimators"]),
        "stage2_best_params": stage2_best_params,
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
        "result_row_count": int(len(cascade_results)),
        "parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_process_run_id": gold_truth.get("process_run_id"),
        "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    },
)

cascade_truth = update_truth_section(
    cascade_truth,
    "artifact_paths",
    {
        "cascade_variant": CASCADE_VARIANT,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_defaults_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_defaults_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_defaults_metadata_path": str(CASCADE_METADATA_PATH),
        "cascade_defaults_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    },
)

cascade_meta_columns = sorted(
    set(
        identify_meta_columns(cascade_results)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

cascade_feature_columns = identify_feature_columns(cascade_results)

cascade_truth_record = build_truth_record(
    truth_base=cascade_truth,
    row_count=len(cascade_results),
    column_count=cascade_results.shape[1] + 3,
    meta_columns=cascade_meta_columns,
    feature_columns=cascade_feature_columns,
)

CASCADE_TRUTH_HASH = cascade_truth_record["truth_hash"]

cascade_results = stamp_truth_columns(
    cascade_results,
    truth_hash=CASCADE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

cascade_truth_path = save_truth_record(
    cascade_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
)

append_truth_index(
    cascade_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

cascade_summary["cascade_truth_hash"] = CASCADE_TRUTH_HASH
cascade_summary["cascade_truth_path"] = str(cascade_truth_path)
cascade_summary["cascade_process_run_id"] = cascade_process_run_id
cascade_summary["gold_truth_hash"] = GOLD_PARENT_TRUTH_HASH
cascade_summary["gold_truth_path"] = str(GOLD_TRUTH_PATH)
cascade_summary["gold_process_run_id"] = gold_truth.get("process_run_id")
cascade_summary["gold_feature_set_id"] = gold_truth_runtime_facts.get("feature_set_id")

cascade_metadata = {
    "cascade_variant": CASCADE_VARIANT,
    "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
    "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
    "cascade_defaults_stage2_selection_mode": STAGE2_SELECTION_MODE,
    "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
    "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
    "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
    "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
    "cascade_defaults_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
    "cascade_defaults_summary_path": str(CASCADE_SUMMARY_PATH),
    "cascade_defaults_metadata_path": str(CASCADE_METADATA_PATH),
    "cascade_defaults_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),

    # upstream truth linkage
    "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(GOLD_TRUTH_PATH),
    "gold_process_run_id": gold_truth.get("process_run_id"),
    "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    "gold_scaler_path": gold_truth_artifact_paths.get("scaler_path"),
    "gold_scaler_kind": gold_truth_runtime_facts.get("scaler_kind_runtime"),
    "gold_recommended_imputation": gold_truth_runtime_facts.get("recommended_imputation"),

    # stage truth linkage
    "cascade_truth_hash": CASCADE_TRUTH_HASH,
    "cascade_truth_path": str(cascade_truth_path),
    "cascade_process_run_id": cascade_process_run_id,
}

cascade_results.to_csv(CASCADE_RESULTS_PATH_CSV, index=False)
cascade_results.to_pickle(CASCADE_RESULTS_PATH_PICKLE)

reference_profile.to_csv(CASCADE_REFERENCE_PROFILE_PATH, index=False)


joblib.dump(stage1_model, STAGE1_MODEL_ARTIFACT_PATH)
joblib.dump(stage1_model, STAGE1_MODELS_PATH)

joblib.dump(stage2_model, STAGE2_MODEL_ARTIFACT_PATH)
joblib.dump(stage2_model, STAGE2_MODELS_PATH)

save_json(cascade_thresholds, CASCADE_THRESHOLDS_PATH)
save_json(cascade_summary, CASCADE_SUMMARY_PATH)
save_json(cascade_metadata, CASCADE_METADATA_PATH)

wandb.save(str(CASCADE_RESULTS_PATH_CSV))
wandb.save(str(CASCADE_RESULTS_PATH_PICKLE))
wandb.save(str(CASCADE_REFERENCE_PROFILE_PATH))
wandb.save(str(STAGE1_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE1_MODELS_PATH))
wandb.save(str(STAGE2_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE2_MODELS_PATH))
wandb.save(str(CASCADE_THRESHOLDS_PATH))
wandb.save(str(CASCADE_SUMMARY_PATH))
wandb.save(str(CASCADE_METADATA_PATH))
wandb.save(str(cascade_truth_path))

ledger.add(
    kind="step",
    step="save_cascade_outputs",
    message="Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.",
    data={
        "cascade_variant": CASCADE_VARIANT,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_defaults_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
        "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_defaults_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_defaults_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_defaults_metadata_path": str(CASCADE_METADATA_PATH),
        "cascade_truth_hash": CASCADE_TRUTH_HASH,
        "cascade_truth_path": str(cascade_truth_path),
        "result_row_count": int(len(cascade_results)),
        "final_alert_count_all_rows": final_cascade_alert_count_all_rows,
        "final_alert_count_test_rows": final_cascade_alert_count_test_rows,
    },
    logger=logger,
)

2026-04-05 05:31:25,478 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_defaults_thresholds.json
2026-04-05 05:31:25,496 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_defaults_summary.json
2026-04-05 05:31:25,514 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_defaults_metadata.json
2026-04-05 05:31:26,016 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:31:26.016455+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'save_cascade_outputs', 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.', 'why': None, 'consequence': None, 'data': {'cascade_variant': 'default', 'stage2_selection_mode': 'fixed', 'cascade_defaults_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_results.csv', 

{'ts_utc': '2026-04-05T05:31:26.016455+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'save_cascade_outputs',
 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.',
 'why': None,
 'consequence': None,
 'data': {'cascade_variant': 'default',
  'stage2_selection_mode': 'fixed',
  'cascade_defaults_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_results.csv',
  'cascade_defaults_results_path_pickle': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_results.pkl',
  'cascade_defaults_reference_profile_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_reference_profile.csv',
  'cascade_defaults_stage1_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_stage1_isolation_forest.joblib',
  'cascade_defaults_stage1_models_path': '/workspace/models/pump/pump__

----

In [ ]:
stage1_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage1_flag",
    row_id_column="meta__row_id",
    score_column="stage1_score",
    decision_column="stage1_decision",
    pred_column="stage1_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage1_detected_rows",
    message="Built the Stage 1 detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage1_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage1_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in stage1_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in stage1_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in stage1_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Stage 1 detected row count: {len(stage1_detected_rows_dataframe):,}")
display(stage1_detected_rows_dataframe.head(20))

2026-04-05 05:31:26,431 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:31:26.431803+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage1_detected_rows', 'message': 'Built the Stage 1 detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage1_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 35615, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Stage 1 detected row count: 35,615


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage1_flag,stage1_score,stage1_decision,stage1_pred,stage2_raw_flag,stage2_flag,cascade_final_flag
0,10421,13040787562538299660,10421,10421,2018-04-08 05:41:00+00:00,asset__001,run__001,NORMAL,0,1,-0.501091,-0.001091,-1,0,0,0
1,10422,17181546761143259301,10422,10422,2018-04-08 05:42:00+00:00,asset__001,run__001,NORMAL,0,1,-0.508270,-0.008270,-1,0,0,0
2,10424,10863625578907073953,10424,10424,2018-04-08 05:44:00+00:00,asset__001,run__001,NORMAL,0,1,-0.509427,-0.009427,-1,0,0,0
3,10425,9940223489881459585,10425,10425,2018-04-08 05:45:00+00:00,asset__001,run__001,NORMAL,0,1,-0.504244,-0.004244,-1,0,0,0
4,10426,2673701136861205737,10426,10426,2018-04-08 05:46:00+00:00,asset__001,run__001,NORMAL,0,1,-0.503254,-0.003254,-1,0,0,0
5,10427,10298021739733576732,10427,10427,2018-04-08 05:47:00+00:00,asset__001,run__001,NORMAL,0,1,-0.501720,-0.001720,-1,0,0,0
6,10428,13817220964081962874,10428,10428,2018-04-08 05:48:00+00:00,asset__001,run__001,NORMAL,0,1,-0.510048,-0.010048,-1,0,0,0
7,10429,17431902560253693386,10429,10429,2018-04-08 05:49:00+00:00,asset__001,run__001,NORMAL,0,1,-0.500433,-0.000433,-1,0,0,0
8,10432,7556830595156906918,10432,10432,2018-04-08 05:52:00+00:00,asset__001,run__001,NORMAL,0,1,-0.504630,-0.004630,-1,0,0,0
9,10433,19320186527523733,10433,10433,2018-04-08 05:53:00+00:00,asset__001,run__001,NORMAL,0,1,-0.501502,-0.001502,-1,0,0,0


In [ ]:
stage2_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage2_flag",
    row_id_column="meta__row_id",
    score_column="stage2_score",
    decision_column="stage2_model_decision",
    pred_column="stage2_model_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage2_model_score",
        "stage2_model_decision",
        "stage2_model_pred",
        "stage2_model_flag",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage2_detected_rows",
    message="Built the Stage 2 detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage2_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage2_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in stage2_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in stage2_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in stage2_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Stage 2 detected row count: {len(stage2_detected_rows_dataframe):,}")
display(stage2_detected_rows_dataframe.head(20))

2026-04-05 05:31:26,930 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:31:26.930078+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage2_detected_rows', 'message': 'Built the Stage 2 detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage2_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 16545, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Stage 2 detected row count: 16,545


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage2_flag,stage2_score,stage2_model_decision,stage2_model_pred,stage1_flag,stage2_raw_flag,stage2_model_score,stage2_model_flag,cascade_final_flag
0,17154,11031252919804255274,17154,17154,2018-04-12 21:54:00+00:00,asset__001,run__001,NORMAL,0,1,0.611132,-0.111132,-1.0,1,1,-0.611132,1.0,1
1,17155,7306247312672969352,17155,17155,2018-04-12 21:55:00+00:00,asset__001,run__001,BROKEN,1,1,0.591552,-0.091552,-1.0,1,1,-0.591552,1.0,1
2,17159,5086261863442700928,17159,17159,2018-04-12 21:59:00+00:00,asset__001,run__001,RECOVERING,1,1,0.588661,-0.088661,-1.0,1,1,-0.588661,1.0,0
3,17163,13012502951683690117,17163,17163,2018-04-12 22:03:00+00:00,asset__001,run__001,RECOVERING,1,1,0.603127,-0.103127,-1.0,1,1,-0.603127,1.0,1
4,17164,18117959071609779637,17164,17164,2018-04-12 22:04:00+00:00,asset__001,run__001,RECOVERING,1,1,0.611455,-0.111455,-1.0,1,1,-0.611455,1.0,1
5,17165,17279807461879477071,17165,17165,2018-04-12 22:05:00+00:00,asset__001,run__001,RECOVERING,1,1,0.603794,-0.103794,-1.0,1,1,-0.603794,1.0,1
6,17166,8494268225196074031,17166,17166,2018-04-12 22:06:00+00:00,asset__001,run__001,RECOVERING,1,1,0.594252,-0.094252,-1.0,1,1,-0.594252,1.0,1
7,17167,4206025446141665590,17167,17167,2018-04-12 22:07:00+00:00,asset__001,run__001,RECOVERING,1,1,0.592128,-0.092128,-1.0,1,1,-0.592128,1.0,1
8,17168,18000093140899971975,17168,17168,2018-04-12 22:08:00+00:00,asset__001,run__001,RECOVERING,1,1,0.602306,-0.102306,-1.0,1,1,-0.602306,1.0,1
9,17169,7831290092567509806,17169,17169,2018-04-12 22:09:00+00:00,asset__001,run__001,RECOVERING,1,1,0.608249,-0.108249,-1.0,1,1,-0.608249,1.0,1


In [ ]:
stage3_evidence_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage3_profile_breach_flag",
    row_id_column="meta__row_id",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage3_profile_breach_count",
        "stage3_secondary_breach_count",
        "stage3_profile_breach_flag",
        "stage3_corroboration_flag",
        "stage3_persistence_flag",
        "stage3_drift_flag",
        "stage3_rule_evidence_count",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage3_evidence_rows",
    message="Built the Stage 3 evidence-row dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage3_profile_breach_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage3_evidence_rows_dataframe)),
    },
    logger=logger,
)

print(f"Stage 3 evidence row count: {len(stage3_evidence_rows_dataframe):,}")
display(stage3_evidence_rows_dataframe.head(20))

2026-04-05 05:31:27,572 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:31:27.572569+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage3_evidence_rows', 'message': 'Built the Stage 3 evidence-row dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage3_profile_breach_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 72702}}


Stage 3 evidence row count: 72,702


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage3_profile_breach_flag,stage1_flag,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_corroboration_flag,stage3_persistence_flag,stage3_drift_flag,stage3_rule_evidence_count,cascade_final_flag
0,775,17087774777964415442,775,775,2018-04-01 12:55:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
1,776,2838568638460419167,776,776,2018-04-01 12:56:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
2,801,12808854361915932714,801,801,2018-04-01 13:21:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
3,812,981528527789039639,812,812,2018-04-01 13:32:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
4,813,16186518969152706661,813,813,2018-04-01 13:33:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
5,814,374737371811597475,814,814,2018-04-01 13:34:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
6,815,17278739453429850848,815,815,2018-04-01 13:35:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
7,829,9684880162965006416,829,829,2018-04-01 13:49:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
8,830,13029152385354111797,830,830,2018-04-01 13:50:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
9,831,5060137085806035708,831,831,2018-04-01 13:51:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0


In [ ]:
final_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="cascade_final_flag",
    row_id_column="meta__row_id",
    score_column="stage2_score",
    decision_column="stage2_model_decision",
    pred_column="stage2_model_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage2_model_score",
        "stage2_model_decision",
        "stage2_model_pred",
        "stage2_model_flag",
        "stage3_profile_breach_count",
        "stage3_secondary_breach_count",
        "stage3_profile_breach_flag",
        "stage3_corroboration_flag",
        "stage3_persistence_flag",
        "stage3_drift_flag",
        "stage3_rule_evidence_count",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_final_detected_rows",
    message="Built the final detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "cascade_final_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(final_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in final_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in final_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in final_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Final cascade detected row count: {len(final_detected_rows_dataframe):,}")
display(final_detected_rows_dataframe.head(20))

2026-04-05 05:31:28,111 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:31:28.111498+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_final_detected_rows', 'message': 'Built the final detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'cascade_final_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 16501, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Final cascade detected row count: 16,501


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,cascade_final_flag,stage2_score,stage2_model_decision,stage2_model_pred,stage1_flag,stage2_raw_flag,stage2_flag,stage2_model_score,stage2_model_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_persistence_flag,stage3_drift_flag,stage3_rule_evidence_count
0,17154,11031252919804255274,17154,17154,2018-04-12 21:54:00+00:00,asset__001,run__001,NORMAL,0,1,0.611132,-0.111132,-1.0,1,1,1,-0.611132,1.0,1,3,0,1,0,1,2
1,17155,7306247312672969352,17155,17155,2018-04-12 21:55:00+00:00,asset__001,run__001,BROKEN,1,1,0.591552,-0.091552,-1.0,1,1,1,-0.591552,1.0,1,3,0,1,1,1,3
2,17163,13012502951683690117,17163,17163,2018-04-12 22:03:00+00:00,asset__001,run__001,RECOVERING,1,1,0.603127,-0.103127,-1.0,1,1,1,-0.603127,1.0,1,3,0,1,0,1,2
3,17164,18117959071609779637,17164,17164,2018-04-12 22:04:00+00:00,asset__001,run__001,RECOVERING,1,1,0.611455,-0.111455,-1.0,1,1,1,-0.611455,1.0,1,3,0,1,1,1,3
4,17165,17279807461879477071,17165,17165,2018-04-12 22:05:00+00:00,asset__001,run__001,RECOVERING,1,1,0.603794,-0.103794,-1.0,1,1,1,-0.603794,1.0,1,2,0,1,1,0,2
5,17166,8494268225196074031,17166,17166,2018-04-12 22:06:00+00:00,asset__001,run__001,RECOVERING,1,1,0.594252,-0.094252,-1.0,1,1,1,-0.594252,1.0,1,2,0,1,1,0,2
6,17167,4206025446141665590,17167,17167,2018-04-12 22:07:00+00:00,asset__001,run__001,RECOVERING,1,1,0.592128,-0.092128,-1.0,1,1,1,-0.592128,1.0,1,2,0,1,1,0,2
7,17168,18000093140899971975,17168,17168,2018-04-12 22:08:00+00:00,asset__001,run__001,RECOVERING,1,1,0.602306,-0.102306,-1.0,1,1,1,-0.602306,1.0,1,3,0,1,1,0,2
8,17169,7831290092567509806,17169,17169,2018-04-12 22:09:00+00:00,asset__001,run__001,RECOVERING,1,1,0.608249,-0.108249,-1.0,1,1,1,-0.608249,1.0,1,3,0,1,1,0,2
9,17170,14785262232998047494,17170,17170,2018-04-12 22:10:00+00:00,asset__001,run__001,RECOVERING,1,1,0.620178,-0.120178,-1.0,1,1,1,-0.620178,1.0,1,3,0,1,1,1,3


----

## Finalize the Ledger and Close the Tracking Run

This step writes the cascade ledger to disk and cleanly closes the experiment tracking run.

By the time I get here, the important modeling and artifact creation work is already complete. So this section is mainly about wrapping up the notebook in a structured way.

In [ ]:
ledger.add(
    kind="step",
    step="finalize_cascade_modeling",
    message="Gold cascade modeling notebook complete.",
    data={
        "cascade_variant": CASCADE_VARIANT,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "cascade_defaults_metrics": cascade_metrics,
        "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
    },
    logger=logger,
)

cascade_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_CASCADE_LEDGER_FILE_NAME
ledger.write_json(cascade_ledger_path)

wandb.save(str(cascade_ledger_path))
wandb_run.finish()

2026-04-05 05:31:28,554 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T05:31:28.554852+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_cascade_modeling', 'message': 'Gold cascade modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'cascade_variant': 'default', 'stage2_selection_mode': 'fixed', 'cascade_defaults_metrics': {'model': '3-Stage Cascade', 'stage1_alert_count_all_rows': 35615, 'stage2_alert_count_all_rows': 16545, 'final_alert_count_all_rows': 16501, 'stage1_alert_count_test_rows': 6215, 'stage2_alert_count_test_rows': 1176, 'final_alert_count_test_rows': 1166, 'precision': 0.05574614065180103, 'recall': 0.5508474576271186, 'f1': 0.10124610591900311}, 'cascade_defaults_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_results.csv', 'cascade_defaults_results_path_pickle': '/workspace/artifacts/gold/pump/pump__gold__cascade_defaults_results.pkl', 'cascade_defaults_st

----

## Run Final Lineage and Consistency Checks

Before I treat the cascade notebook as complete, I run a final sanity check on the saved cascade results and truth artifacts.

This check verifies things like:
- required lineage columns exist in the results dataframe
- the dataframe truth hash matches the saved cascade truth
- the parent truth hash matches the Gold parent truth
- the saved truth file exists
- the saved metadata points back to the correct cascade truth hash

I like ending with this because it confirms that the cascade output is not only saved, but also internally consistent and traceable.

In [ ]:
required_cascade_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_cascade_meta_columns = [
    column_name
    for column_name in required_cascade_meta_columns
    if column_name not in cascade_results.columns
]
if missing_cascade_meta_columns:
    raise ValueError(
        f"cascade_results is missing required lineage columns: {missing_cascade_meta_columns}"
    )

cascade_results_truth_hash_check = extract_truth_hash(cascade_results)
if cascade_results_truth_hash_check is None:
    raise ValueError("cascade_results does not contain a readable meta__truth_hash value.")

if cascade_results_truth_hash_check != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_results truth hash does not match CASCADE_TRUTH_HASH:\n"
        f"dataframe={cascade_results_truth_hash_check}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

cascade_parent_values = cascade_results["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not cascade_parent_values:
    raise ValueError("cascade_results is missing populated meta__parent_truth_hash values.")

if len(cascade_parent_values) != 1:
    raise ValueError(f"cascade_results has multiple parent truth hashes: {cascade_parent_values}")

if cascade_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "cascade_results parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={cascade_parent_values[0]}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

if not Path(cascade_truth_path).exists():
    raise FileNotFoundError(f"Cascade truth file was not created: {cascade_truth_path}")

loaded_cascade_truth = load_json(cascade_truth_path)

if loaded_cascade_truth.get("truth_hash") != CASCADE_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file hash does not match CASCADE_TRUTH_HASH:\n"
        f"file={loaded_cascade_truth.get('truth_hash')}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

if loaded_cascade_truth.get("parent_truth_hash") != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_cascade_truth.get('parent_truth_hash')}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

saved_cascade_metadata = load_json(CASCADE_METADATA_PATH)
if saved_cascade_metadata.get("cascade_truth_hash") != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_metadata cascade_truth_hash does not match CASCADE_TRUTH_HASH:\n"
        f"metadata={saved_cascade_metadata.get('cascade_truth_hash')}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

print("Gold Cascade lineage sanity check passed.")

2026-04-05 05:32:19,359 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/gold_cascade/pump__gold_cascade__truth__de2f210c6130856e2b88eaa5a247edb789e308d0056d2cbd63c1e1fcdb8c2372.json
2026-04-05 05:32:19,371 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_defaults_metadata.json


Gold Cascade lineage sanity check passed.


### Ask

What does this final sanity check really confirm?

### Answer

It confirms that the cascade results can be trusted as a pipeline artifact, not just as notebook output.

A modeling notebook can run successfully and still leave behind broken lineage or mismatched truth metadata. This final check helps guard against that by making sure the dataframe, truth record, and saved metadata all agree with each other.

So this is really a trust check more than a completion check.

----

## Optional PostgreSQL Write for Cascade Outputs

This section is for writing the cascade results into PostgreSQL.

I am treating this as an optional persistence step rather than part of the core cascade modeling deliverable. The main outputs from this notebook are still the saved cascade results, model artifacts, thresholds, summary, metadata, ledger, and truth record. Writing to SQL is useful when I want the cascade artifact available for querying, validation, or downstream integration work.

SQL_SCHEMAS = {
    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
    "synthetic": "synthetic",
    "truth": "truth",
    "audit": "audit",
}

GOLD_ARTIFACT_TABLES = {
    "baseline_results": "baseline_results",
    "baseline_summary": "baseline_summary",
    "baseline_thresholds": "baseline_thresholds",
    "cascade_results": "cascade_results",
    "cascade_summary": "cascade_summary",
    "cascade_thresholds": "cascade_thresholds",
    "comparison_summary": "comparison_summary",
}

SYNTHETIC_ARTIFACT_TABLES = {
    "batch": "batch",
    "stream": "stream",
    "sensor_messages": "sensor_messages",
}




GOLD_SCHEMA = SQL_SCHEMAS["gold"]

engine = get_engine_from_env()



gold_cascade_results_sql = prepare_layer_dataframe(
    cascade_results,
    truth_hash=CASCADE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
    process_run_id=GOLD_PROCESS_RUN_ID,
    add_loaded_at_column=True,
)

gold_cascade_results_table = write_layer_dataframe(
    engine=engine,
    dataframe=gold_cascade_results_sql,
    schema=GOLD_SCHEMA,
    dataset_name=DATASET_NAME,
    artifact_name=GOLD_ARTIFACT_TABLES["cascade_results"],
    if_exists="replace",
    index=False,
)

print(f"Wrote table: {GOLD_SCHEMA}.{gold_cascade_results_table}")